# Lecture: Relationships in Space and Time

Time and location add structure to data. We will use candy production, Gapminder country summaries, Montréal car-sharing availability, and Hurricane Ian to explore relationships across time and geography.

## Learning goals

- Identify trends, seasonality, and dependence between observations close together in time.
- Compare relationships across years, countries, and population sizes.
- Choose between a relationship plot and a geographic map based on the question.
- Diagnose an out-of-order animation and correct its chronological sequence.
- Relate hurricane pressure and wind speed to intensity while recognizing what these measurements leave out.

## The reproducible analysis workflow

1. **Question:** What do we want to learn?
2. **Data:** Which observations and features can help answer it?
3. **Evidence:** What should we calculate and display?
4. **Check:** Does the evidence make sense in context and agree with the original records?
5. **Conclusion:** What claim is supported?
6. **Limitation:** What should we avoid concluding?

Each example follows the same six steps. We will predict a result, inspect the data, create figures, and connect conclusions to checked evidence.

## Interactive visualization with Plotly Express

Plotly Express, imported as `px`, creates interactive figures with hover information, zooming, legends, and animation. Pandas helps us read and organize the data.

Keep axes, colors, and marker-size scales consistent across animation frames. Pause and hover to compare values. Geographic outlines and background map tiles may require internet access; the observations come from local CSV files.

## Import the libraries

In [ ]:
import pandas as pd
import plotly.express as px

# Part 1: Candy production over time

The [Federal Reserve's FRED series IPG3113N](https://fred.stlouisfed.org/series/IPG3113N) measures **U.S. sugar and confectionery production**, not candy prices or sales. It is a monthly index with 2017 = 100 and is **not seasonally adjusted**. The local snapshot contains January 1972–August 2026, retrieved September 30, 2026; later releases can revise historical values.

`observation_date` identifies the month. `IPG3113N` is the production index. An index of 110 means production is 10% above the 2017 reference level; it is not a dollar amount.

## Step 1: Question

**How has U.S. candy production changed over time?**

### Discussion

**Predict whether production will follow a steady trend, a repeating yearly pattern, or both. Explain your reasoning.**

## Step 2: Data

Read the monthly production index and parse `observation_date` as a datetime. Sort the records so that lines connect months chronologically. Inspect the beginning and end of the file, its dimensions, storage types, and missing values before selecting an analysis period.

Load, sort, and inspect the candy data.

In [ ]:
candy_df = pd.read_csv('data/candy_production.csv', parse_dates=['observation_date'])
candy_df = candy_df.sort_values('observation_date').reset_index(drop=True)

In [ ]:
print(candy_df.head())

In [ ]:
print(candy_df.tail())

In [ ]:
print(candy_df.shape)

In [ ]:
candy_df.info()

In [ ]:
print(candy_df.isna().sum())

### Discussion: check the data

Use the definitions and inspection output to classify each feature. Dates and hour labels are temporal; string and datetime storage types may vary with the Pandas version.

| Feature | Feature type | Data storage type | Missing values |
| --- | --- | --- | --- |
| `observation_date` | | | |
| `IPG3113N` | | | |

### Discussion

**What does one row represent? What period is covered, and how many records were removed from the dataset?**

## Step 3: Evidence

A **time-series plot** places time on the horizontal axis and the production index on the vertical axis. Connecting consecutive monthly observations helps us follow changes over time. Zoom into shorter periods to inspect the pattern more closely.

Plot the complete monthly series.

In [ ]:
fig = px.line(candy_df, x='observation_date', y='IPG3113N',
              title='U.S. sugar and confectionery production, 1972–2026',
              labels={'observation_date':'Month', 'IPG3113N':'Production index (2017 = 100)'})
fig.show()

## Step 4: Check

Use hover to check that the dates advance in time order as you move along the line. Hover over particularly high and low points and record when they occurred. Can you identify world events, such as the COVID-19 pandemic, that coincided with those dates? The timing suggests questions to investigate; it does not establish what caused a change.

## Step 5: Conclusion

**How has U.S. candy production changed over time?**

Production rises and falls over time, with recurring patterns and some unusually large changes. Hovering connects those changes to particular months and helps us ask whether they coincided with major events.

## Step 6: Limitation

**Seasonality** means a pattern repeats at similar times of year. **Autocorrelation** means observations near one another in time tend to be related.

A single straight line cannot follow the recurring ups and downs in candy production, and a basic model that treats each month as independent overlooks time dependence.

### Discussion

What patterns in this figure would a straight line miss?

There are specialized **time-series models** for time-ordered data. They can represent seasonal patterns and dependence between observations. We will not cover their details in this lecture.

# Part 2: Life expectancy, income, population, and geography

Use `data/gapminder.csv`, the [Gapminder teaching dataset](https://jennybc.github.io/gapminder/reference/gapminder.html) distributed with Plotly. It contains 142 countries at five-year intervals from **1952 through 2007**, giving each animation 12 frames.

- `country`, `continent`: geographic labels.
- `year`: observation year.
- `lifeExp`: life expectancy at birth, in years.
- `gdpPercap`: GDP per capita in inflation-adjusted, purchasing-power-adjusted international dollars in this historical dataset.
- `pop`: population count.
- `iso_alpha`, `iso_num`: country identifiers. Numeric country codes are categorical identifiers, not measurements.

Each row represents one country in one year. This historical subset does not include every country; an absent country should not be interpreted as having zero life expectancy.

## Step 1: Question

**How are life expectancy and GDP per capita related, and how does that relationship vary over time, geography, and population size?**

### Discussion

**Predict the direction of the GDP–life expectancy relationship and whether all countries will change in the same way over time.**

## Step 2: Data

Read and inspect the country-year data, including storage types, missing values, and coverage. Both figures use the same observations.

Load and inspect the Gapminder data.

In [ ]:
gap_df = pd.read_csv('data/gapminder.csv')

In [ ]:
print(gap_df.head())

In [ ]:
print(gap_df.tail())

In [ ]:
print(gap_df.shape)

In [ ]:
gap_df.info()

In [ ]:
print(gap_df.isna().sum())

In [ ]:
print('Countries:', gap_df['country'].nunique())

In [ ]:
print('Years:', sorted(gap_df['year'].unique()))

### Discussion: check the data

Classify the features using their definitions and the inspection output.

| Feature | Feature type | Data storage type | Missing values |
| --- | --- | --- | --- |
| `country` | | | |
| `continent` | | | |
| `year` | | | |
| `lifeExp` | | | |
| `pop` | | | |
| `gdpPercap` | | | |
| `iso_alpha` | | | |
| `iso_num` | | | |

### Discussion

**What does one row represent, and how many records were removed from the dataset?**

In [ ]:
gap_df.sort_values(['year', 'country'], inplace=True)

## Step 3: Evidence

Use a Plotly Express bubble plot to explore the relationship between GDP per capita and life expectancy. Population controls bubble area, continent controls color, and the animation advances by year. A logarithmic GDP axis makes proportional income differences easier to compare. Then use an animated choropleth to examine life expectancy in geographic context.

Both figures use the same country-year observations and fixed scales across years. Each bubble represents one country or territory, and its area represents population.

In [ ]:
years = sorted(gap_df['year'].unique())

Create the animated bubble plot with fixed axes and Plotly’s default continent colors.

### Add time and population

In `px.scatter()`, `animation_frame="year"` creates a frame for each year, `animation_group="country"` keeps track of country identity, `size="pop"` maps population to bubble area, and `color="continent"` distinguishes broad continents. `log_x=True` changes the display scale while hover labels retain GDP values in their original units.

In [ ]:
fig = px.scatter(gap_df, x='gdpPercap', y='lifeExp',
                 animation_frame='year', animation_group='country',
                 size='pop', color='continent', hover_name='country', log_x=True,
                 range_x=[200, 120000], range_y=[20, 90], size_max=55,
                 category_orders={'year':years},
                 labels={'gdpPercap':'GDP per capita (historical international dollars)',
                         'lifeExp':'Life expectancy at birth (years)', 'pop':'Population', 'continent':'Continent'},
                 title='Life expectancy, income, and population, 1952–2007')
fig.update_layout(height=500)
fig.show()

Create the animated life-expectancy choropleth with a fixed color range.

### Show the geographic pattern

A **choropleth** shades geographic regions according to a measurement. `locations="iso_alpha"` matches country codes to boundaries. The life-expectancy color scale stays fixed from 20 to 90 years. An uncolored place has no displayed value in that frame; it does not have zero life expectancy.

In [ ]:
fig = px.choropleth(gap_df, locations='iso_alpha', color='lifeExp',
                    hover_name='country', hover_data=['gdpPercap', 'pop'],
                    animation_frame='year', animation_group='country',
                    category_orders={'year':years}, range_color=[20, 90],
                    color_continuous_scale='Viridis', projection='natural earth',
                    labels={'lifeExp':'Life expectancy (years)',
                            'gdpPercap':'GDP per capita (historical international dollars)', 'pop':'Population'},
                    title='Life expectancy by country, 1952–2007')
fig.update_geos(showcountries=True, showland=True, landcolor='lightgray')
fig.update_layout(height=500, 
    margin={'l':10, 'r':30, 't':60, 'b':90},
    font={'size':16},
    coloraxis_colorbar={
        'title':{'text':'Life expectancy<br>(years)', 'font':{'size':18}},
        'tickfont':{'size':16}, 'thickness':32, 'len':0.85
    }
)
fig.show()

## Step 4: Check

Pause both animations at 1952 and 2007. Hover over the same two countries in each figure and compare their values. Confirm that colors and axes retain their meaning across years. Look for places without displayed data on the map. Would you have noticed their absence from the bubble plot?

### Discussion

**How did your two countries change? Which display made it easier to notice missing countries or territories?**

## Step 5: Conclusion

**How are life expectancy and GDP per capita related, and how does that relationship vary over time, geography, and population size?**

Higher GDP per capita generally accompanies longer life expectancy, but countries with similar incomes can have different outcomes. Many countries move toward higher income and longer life expectancy over time, with setbacks and different rates of change. Population-sized bubbles show how many people a country-level estimate represents; the map shows geographic patterns.

## Step 6: Limitation

### Discussion

Which parts of the question are easier or harder to answer from the bubble plot and from the choropleth? Consider the quantitative relationship, geography, population, change over time, and countries with no displayed data.

A country with no plotted data leaves no obvious empty space in a scatterplot, while its geographic area can remain uncolored on a map. Russia is one example in this dataset.

# Part 3: Montréal availability by peak hour

Use `data/montreal_carshare.csv`, the original [Plotly car-sharing demonstration dataset](https://plotly.com/python-api-reference/generated/plotly.express.data.html). Each row summarizes availability near the centroid of a Montréal zone over a month-long period.

- `centroid_lat`, `centroid_lon`: zone-center coordinates in degrees.
- `car_hours`: the supplied car-hours availability measure.
- `peak_hour`: the supplied peak-hour label, 0–23.

A zone appears once, with one peak-hour label; these are not repeated observations of every zone throughout a single day.

## Step 1: Question

**How does car-sharing availability vary across Montréal and peak hours?**

### Discussion

**Do you expect all areas of the city to have similar availability and peak hours? Why?**

## Step 2: Data

Read and inspect the Montréal availability data. Each row summarizes a zone and its supplied peak-hour label.

Load and inspect the data.

In [ ]:
carshare_df = pd.read_csv('data/montreal_carshare.csv')

In [ ]:
print(carshare_df.head())

In [ ]:
print(carshare_df.tail())

In [ ]:
print(carshare_df.shape)

In [ ]:
carshare_df.info()

In [ ]:
print(carshare_df.isna().sum())

### Discussion: check the data

Use the definitions and inspection output to classify each feature. Dates and hour labels are temporal; string and datetime storage types may vary with the Pandas version.

| Feature | Feature type | Data storage type | Missing values |
| --- | --- | --- | --- |
| `centroid_lat` | | | |
| `centroid_lon` | | | |
| `car_hours` | | | |
| `peak_hour` | | | |

### Discussion

**What does one row represent, and how many records were removed from the dataset?**

## Step 3: Evidence

Create an animated map with car-hours represented by marker size and color. Play the animation and explore the figure.

In [ ]:
fig = px.scatter_map(carshare_df, lat='centroid_lat', lon='centroid_lon',
                     size='car_hours', color='car_hours', animation_frame='peak_hour',
                     range_color=[0, carshare_df['car_hours'].max()],
                     color_continuous_scale='Viridis', size_max=35,
                     center={'lat':45.52, 'lon':-73.62}, zoom=9,
                     map_style='open-street-map',
                     labels={'car_hours':'Car-hours availability', 'peak_hour':'Peak hour'},
                     title='Montréal car-sharing availability')
fig.update_layout(height=500, margin={'l':10, 'r':30, 't':60, 'b':80})
fig.show()

### Discussion

**What do you notice about the animation? Is anything making it difficult to interpret? Inspect the data and suggest a correction.**

Sort the existing DataFrame and recreate the map.

### Put the hour labels in chronological order

Plotly encounters frame labels in DataFrame order. `.sort_values("peak_hour", inplace=True)` rearranges the existing DataFrame chronologically; it does not create a second DataFrame or change any measurements.

In [ ]:
carshare_df.sort_values('peak_hour', inplace=True)
fig = px.scatter_map(carshare_df, lat='centroid_lat', lon='centroid_lon',
                     size='car_hours', color='car_hours', animation_frame='peak_hour',
                     range_color=[0, carshare_df['car_hours'].max()],
                     color_continuous_scale='Viridis', size_max=35,
                     center={'lat':45.52, 'lon':-73.62}, zoom=9,
                     map_style='open-street-map',
                     labels={'car_hours':'Car-hours availability', 'peak_hour':'Peak hour'},
                     title='Montréal car-sharing availability')
fig.update_layout(height=500, margin={'l':10, 'r':30, 't':60, 'b':80})
fig.show()

## Step 4: Check

Play the revised animation and check that the hour labels now advance from 0 to 23. Hover over a few markers to check their availability values. What changed, and what stayed the same?

## Step 5: Conclusion

**How does car-sharing availability vary across Montréal and peak hours?**

The figure shows where zones with different peak-hour labels are located and how their car-hours availability differs. Sorting the hours makes this sequence interpretable. Each frame shows zones assigned that peak hour, not the movement of individual cars or repeated observations of all zones.

## Step 6: Limitation

### Discussion: what can we conclude?

Can this animation show how availability at one zone changes throughout a day? Explain what a zone missing from a particular frame means, and identify what additional observations would be needed to study availability throughout a day.

# Part 4: Hurricane Ian’s pressure and winds

Use `data/hurricane_ian_2022.csv`, a course extract from [NOAA’s HURDAT2 archive](https://www.nhc.noaa.gov/data/#hurdat). It contains 36 observations of Ian’s tropical stages during September 23–30, 2022, including additional landfall observations.

- `observation_time`: observation date and time in UTC.
- `maximum_sustained_wind_knots`: maximum sustained wind speed, in knots.
- `minimum_central_pressure_mb`: minimum central pressure, in millibars.
- `record_identifier`: `L` marks landfall; blank means no special event marker.
- `status`: TD = tropical depression, TS = tropical storm, HU = hurricane.

Here, severity means physical intensity. Wind and pressure do not describe every hazard or the resulting damage.

## Step 1: Question

**How are central pressure and maximum sustained wind speed related to Hurricane Ian’s severity as it changes over time and location?**

### Discussion

**Predict how pressure and wind speed will change as the hurricane becomes more intense.**

## Step 2: Data

Read the observations, parse `observation_time`, and sort chronologically. Inspect the dimensions, storage types, and missing values. Blank event markers mean no special event code; they do not make wind or pressure measurements unusable.

Load, sort, and inspect Ian’s observations.

In [ ]:
ian_df = pd.read_csv('data/hurricane_ian_2022.csv', parse_dates=['observation_time'])
ian_df = ian_df.sort_values('observation_time').reset_index(drop=True)

In [ ]:
print(ian_df.head())

In [ ]:
print(ian_df.tail())

In [ ]:
print(ian_df.shape)

In [ ]:
ian_df.info()

In [ ]:
print(ian_df.isna().sum())

### Discussion: check the data

Use the definitions and inspection output to classify the listed features. Dates and hour labels are temporal; string and datetime storage types may vary with the Pandas version.

| Feature | Feature type | Data storage type | Missing values |
| --- | --- | --- | --- |
| `storm_id` | | | |
| `storm_name` | | | |
| `observation_time` | | | |
| `status` | | | |
| `record_identifier` | | | |
| `maximum_sustained_wind_knots` | | | |
| `minimum_central_pressure_mb` | | | |

### Discussion

**What does one row represent? How many records were removed from the dataset, and why should blank event markers remain?**

## Step 3: Evidence

Animate Ian’s location over time. Marker **area represents wind speed** and **color represents central pressure**. The scales stay fixed: larger markers mean stronger winds, and redder markers mean lower pressure.

Pause the animation and hover over the storm to read both measurements. Pay particular attention to the landfall frames. The marker represents storm intensity, not the geographic size of the hurricane.

In [ ]:
ian_df['time_utc'] = ian_df['observation_time'].dt.strftime('%Y-%m-%d %H:%M UTC')
ian_df['event'] = ian_df['record_identifier'].fillna('No event marker').replace({'L':'Landfall'})
fig = px.scatter_geo(ian_df, lat='latitude', lon='longitude',
                     animation_frame='time_utc', animation_group='storm_id',
                     size='maximum_sustained_wind_knots', size_max=45,
                     color='minimum_central_pressure_mb', range_color=[930, 1010],
                     color_continuous_scale='YlOrRd_r',
                     hover_name='storm_name',
                     hover_data={'latitude':False, 'longitude':False, 'storm_id':False,
                                 'maximum_sustained_wind_knots':True,
                                 'minimum_central_pressure_mb':True, 'event':True},
                     labels={'maximum_sustained_wind_knots':'Wind (knots)',
                             'minimum_central_pressure_mb':'Pressure (mb)',
                             'time_utc':'Observation time', 'event':'Event'},
                     projection='mercator',
                     title='Hurricane Ian: wind speed and central pressure')
fig.update_geos(
    fitbounds=False, lataxis_range=[11, 36], lonaxis_range=[-89, -63],
    resolution=50, showland=True, landcolor='whitesmoke',
    showocean=True, oceancolor='aliceblue',
    showcountries=True, showsubunits=True
)
fig.update_layout(height=500, margin={'l':10, 'r':30, 't':60, 'b':100})
fig.show()

## Step 4: Check

Pause and hover over the frames before, during, and after landfall. Use the event label to identify landfall observations. What happens to wind speed and pressure as Ian moves onto and across Florida? Compare the marker size and color with their hover values.

## Step 5: Conclusion

**How are central pressure and maximum sustained wind speed related to Hurricane Ian’s severity as it changes over time and location?**

As Ian approaches southwest Florida, the map shows strong winds with low pressure. Hover values at and after landfall show winds weakening while pressure rises as the storm crosses Florida. The animation connects these changes with the storm’s movement; one measurement or one frame does not describe its whole life cycle.

## Step 6: Limitation

### Discussion

What can this animated map tell us about Ian’s intensity, and what does it leave out about hurricane severity and impacts?

## Bringing the examples together

- Candy production introduces seasonal patterns and time dependence, with hover linking unusual values to dates.
- Gapminder compares income and life expectancy across years and shows how maps reveal geographic patterns and missing places.
- The Montréal example shows why we must inspect a figure and check its sequence before interpreting it.
- Ian’s animation uses size, color, and hover to connect pressure and wind changes with movement and landfall.

### Discussion

Choose one example. How did hover information help you check or refine your interpretation?